# 00.01 · De una tabla de Excel a información confiable

**Objetivo:** importar tres CSV en Excel, hacer controles de calidad y validar los cálculos con pandas. La actividad de Excel se realiza en el programa de hoja de cálculo; este notebook facilita una comprobación independiente.

**Reto:** ¿qué errores en la información harían que una decisión comercial fuera equivocada?

In [ ]:
from pathlib import Path
import pandas as pd
base = Path('../datos') if Path('../datos/ventas.csv').exists() else Path('00 Fundamentos de Datos - Excel y SQL/datos')
ventas = pd.read_csv(base / 'ventas.csv', parse_dates=['fecha'])
clientes = pd.read_csv(base / 'clientes.csv')
productos = pd.read_csv(base / 'productos.csv')
print(ventas.shape, clientes.shape, productos.shape)
ventas.head()

## En Excel (hazlo tú)
1. Importa los CSV desde **Datos → Obtener datos** o abre cada CSV y conviértelo en tabla.
2. Comprueba formatos, duplicados de ID y claves ajenas sin correspondencia.
3. Usa BUSCARX / Power Query para asignar cliente, país y producto a cada venta.
4. Agrega columnas *ingresos*, *costo variable*, *margen bruto* y *porcentaje de margen*.
5. Construye tablas dinámicas por mes/país/producto. Compara con los cálculos siguientes.

In [ ]:
assert not ventas.id.duplicated().any(), 'IDs de venta duplicados'
assert ventas.cliente_id.isin(clientes.cliente_id).all(), 'Cliente desconocido'
assert ventas.producto_id.isin(productos.producto_id).all(), 'Producto desconocido'
assert (ventas[['unidades','precio_unitario','costo_unitario']] >= 0).all().all()
assert ventas.notna().all().all()
df = ventas.merge(clientes, on='cliente_id', validate='many_to_one').merge(productos,on='producto_id',validate='many_to_one')
df['ingresos'] = df.unidades * df.precio_unitario
df['costo_variable'] = df.unidades * df.costo_unitario
df['margen_bruto'] = df.ingresos - df.costo_variable
df['mes'] = df.fecha.dt.to_period('M').astype(str)
df.head()

In [ ]:
resumen = df.groupby(['pais','producto'],as_index=False).agg(ingresos=('ingresos','sum'),costo=('costo_variable','sum'),margen=('margen_bruto','sum'),operaciones=('id','count'))
resumen['margen_pct'] = (100*resumen.margen/resumen.ingresos).round(2)
print('Ventas:', len(df), '| Ingresos:', round(df.ingresos.sum(),2), '| Margen bruto:',round(df.margen_bruto.sum(),2))
resumen.sort_values('margen',ascending=False)

## Escenarios: sensibilidad comercial
El análisis **no incluye** fletes, aranceles, sueldos ni estructura fija. Los escenarios son supuestos didácticos, no predicciones.

In [ ]:
escenarios = pd.DataFrame({'escenario':['base','insumos +10%','descuento 5%','transporte +7% sobre costo'], 'ingreso':[df.ingresos.sum(),df.ingresos.sum(),df.ingresos.sum()*.95,df.ingresos.sum()], 'costo':[df.costo_variable.sum(),df.costo_variable.sum()*1.10,df.costo_variable.sum(),df.costo_variable.sum()*1.07]})
escenarios['margen_bruto_ajustado'] = escenarios.ingreso - escenarios.costo
escenarios

## Entrega larga
En Excel prepara 3 tablas dinámicas, dos gráficos, controles de calidad, análisis de sensibilidad, una hoja de supuestos y una conclusión de hasta 400 palabras. Explica cuándo **no** usarías estos resultados. **Extensión:** agrega una fila duplicada de forma deliberada y demuestra cómo detectarla.